# 아로코 전체 상품 스크래핑

브랜드 전체 상품 목록에서 상품 ID, 상품명, 가격, 품절 여부, 상세 URL, 이미지 URL을 수집합니다. 상품 정보가 HTML에 포함되어 있어 Selenium 없이 `requests`와 `BeautifulSoup`를 사용합니다.

> 공개 상품만 천천히 요청합니다. 이용약관과 robots 정책을 확인하고 요청 간격을 지나치게 줄이지 마세요.

In [1]:
import re
import time
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
import urllib3
from bs4 import BeautifulSoup

BASE_URL = "https://arokor.com"
LIST_URL = f"{BASE_URL}/product/list.html"
CATEGORY_NO = 23  # 브랜드 전체 상품
MAX_PAGES = 50
REQUEST_DELAY = 1.0
VERIFY_SSL = False  # 현재 교육 환경의 인증서 체인 오류 때문에 비활성화
if not VERIFY_SSL:
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

session = requests.Session()
session.headers.update({
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                   "AppleWebKit/537.36 Chrome/140.0 Safari/537.36"),
    "Accept-Language": "ko-KR,ko;q=0.9,en;q=0.8",
    "Referer": f"{BASE_URL}/",
})

## 상품 카드 파싱

`anchorBoxId_상품번호`와 `.description`을 기준으로 값을 읽고 상대 주소를 절대 주소로 바꿉니다.

In [2]:
def absolute_url(value):
    if not value:
        return None
    return "https:" + value if value.startswith("//") else urljoin(BASE_URL, value)


def parse_product_card(card, page):
    name_link = card.select_one(".description .name a")
    name_spans = name_link.select(":scope > span") if name_link else []
    image = card.select_one(".thumbnail .prdImg img")
    description = card.select_one(".description")
    id_match = re.search(r"anchorBoxId_(\d+)", card.get("id", ""))
    price_text = description.get("ec-data-price", "") if description else ""
    price_digits = re.sub(r"[^0-9]", "", price_text)

    return {
        "product_id": int(id_match.group(1)) if id_match else None,
        "name": name_spans[-1].get_text(" ", strip=True) if name_spans else None,
        "price": int(price_digits) if price_digits else None,
        "sold_out": card.select_one('img[alt*="품절"], .soldout_icon') is not None,
        "product_url": absolute_url(name_link.get("href")) if name_link else None,
        "image_url": absolute_url(image.get("src")) if image else None,
        "page": page,
    }


def fetch_product_page(page):
    response = session.get(
        LIST_URL, params={"cate_no": CATEGORY_NO, "page": page},
        timeout=20, verify=VERIFY_SSL,
    )
    response.raise_for_status()
    response.encoding = "utf-8"
    soup = BeautifulSoup(response.text, "lxml")
    cards = soup.select('ul.prdList > li[id^="anchorBoxId_"]')
    return [parse_product_card(card, page) for card in cards]

## 전체 페이지 수집

상품 카드가 없는 페이지에서 멈추고 상품 ID 기준으로 중복을 제거합니다.

In [3]:
products = []

for page in range(1, MAX_PAGES + 1):
    page_products = fetch_product_page(page)
    if not page_products:
        print(f"{page}페이지: 상품 없음 → 수집 종료")
        break
    products.extend(page_products)
    print(f"{page}페이지: {len(page_products)}개 (누적 {len(products)}개)")
    time.sleep(REQUEST_DELAY)
else:
    print(f"MAX_PAGES={MAX_PAGES}에 도달했습니다. 필요하면 늘리세요.")

df = (pd.DataFrame(products)
      .dropna(subset=["product_id", "name"])
      .drop_duplicates(subset="product_id", keep="first")
      .sort_values("product_id")
      .reset_index(drop=True))

print(f"최종 수집 상품: {len(df):,}개")
df.head()

1페이지: 12개 (누적 12개)
2페이지: 12개 (누적 24개)
3페이지: 12개 (누적 36개)
4페이지: 12개 (누적 48개)
5페이지: 12개 (누적 60개)
6페이지: 12개 (누적 72개)
7페이지: 12개 (누적 84개)
8페이지: 12개 (누적 96개)
9페이지: 12개 (누적 108개)
10페이지: 12개 (누적 120개)
11페이지: 12개 (누적 132개)
12페이지: 12개 (누적 144개)
13페이지: 12개 (누적 156개)
14페이지: 12개 (누적 168개)
15페이지: 12개 (누적 180개)
16페이지: 12개 (누적 192개)
17페이지: 12개 (누적 204개)
18페이지: 12개 (누적 216개)
19페이지: 12개 (누적 228개)
20페이지: 12개 (누적 240개)
21페이지: 12개 (누적 252개)
22페이지: 12개 (누적 264개)
23페이지: 12개 (누적 276개)
24페이지: 12개 (누적 288개)
25페이지: 12개 (누적 300개)
26페이지: 12개 (누적 312개)
27페이지: 12개 (누적 324개)
28페이지: 12개 (누적 336개)
29페이지: 12개 (누적 348개)
30페이지: 12개 (누적 360개)
31페이지: 9개 (누적 369개)
32페이지: 상품 없음 → 수집 종료
최종 수집 상품: 369개


,product_id,name,price,sold_out,product_url,image_url,page
0,13,STEPHANE HUMBERT LUCAS 2022 제너레이션 EAU DE PARFUM,395000,False,https://arokor.com/product/stephane-humbert-lu...,https://arokor.com/web/product/medium/202306/5...,31
1,14,STEPHANE HUMBERT LUCAS 블랙 젬스톤 EAU DE PARFUM,395000,True,https://arokor.com/product/stephane-humbert-lu...,https://arokor.com/web/product/medium/202306/2...,31
2,15,STEPHANE HUMBERT LUCAS 크라잉 오브 이블 EAU DE PARFUM,320000,False,https://arokor.com/product/stephane-humbert-lu...,https://arokor.com/web/product/medium/202306/3...,31
3,16,STEPHANE HUMBERT LUCAS 갓 오브 파이어 EAU DE PARFUM,320000,False,https://arokor.com/product/stephane-humbert-lu...,https://arokor.com/web/product/medium/202307/d...,31
4,17,STEPHANE HUMBERT LUCAS 이스라 & 미라즈 EAU DE PARFUM...,395000,False,https://arokor.com/product/stephane-humbert-lu...,https://arokor.com/web/product/medium/202306/b...,31


## 검증 및 CSV 저장

누락·중복·품절 건수를 확인하고 Excel에서도 한글이 깨지지 않도록 UTF-8 BOM으로 저장합니다.

In [4]:
display(df.isna().sum().rename("missing_count"))
print("중복 상품 ID:", df["product_id"].duplicated().sum())
print("품절 상품:", int(df["sold_out"].sum()))

data_dir = Path("../data") if Path.cwd().name == "chapt999" else Path("data")
data_dir.mkdir(parents=True, exist_ok=True)
output_path = data_dir / "arokor_products.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"저장 완료: {output_path.resolve()}")

product_id     0
name           0
price          0
sold_out       0
product_url    0
image_url      0
page           0
Name: missing_count, dtype: int64

중복 상품 ID: 0
품절 상품: 150
저장 완료: D:\SourceBank\ai-datapipeline-2026\data\arokor_products.csv
